# 📊 Análisis Exploratorio de Datos (EDA) - Customer Support

Este notebook presenta un Análisis Exploratorio de Datos exhaustivo sobre el dataset de Customer Support. 

**Objetivos:**
1. Comprender la estructura y calidad de los datos.
2. Analizar distribuciones univariadas y bivariadas.
3. Identificar patrones de sentimiento, urgencia e intenciones.
4. Evaluar la temporalidad y volumen de soporte.

## 1. Importación de Librerías y Configuración
Cargamos las librerías necesarias para el análisis y visualización, estableciendo un estilo global.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (14, 6)

## 2. Carga de Datos
Utilizamos la muestra de 20,000 registros estratificados para realizar el análisis exploratorio de forma ágil pero representativa.

In [ ]:
data_path = "../data/customer_support_sample.csv"

try:
    df = pd.read_csv(data_path, parse_dates=["timestamp"])
    print(f"✅ Datos cargados exitosamente: {df.shape[0]:,} filas y {df.shape[1]} columnas.")
except FileNotFoundError:
    print(f"❌ Archivo no encontrado en {data_path}. Asegúrate de ejecutar el script de muestreo primero.")

df.head()

## 3. Resumen Estructural y Calidad de Datos (Data Quality)
Evaluamos los tipos de datos, valores nulos y cardinalidad de las variables categóricas.

In [ ]:
def data_quality_report(df):
    report = pd.DataFrame({
        "Tipo": df.dtypes,
        "Nulos": df.isnull().sum(),
        "% Nulos": (df.isnull().sum() / len(df) * 100).round(2),
        "Valores Únicos": df.nunique()
    })
    return report.sort_values("% Nulos", ascending=False)

data_quality_report(df)

## 4. Análisis Temporal (Volumen a lo largo del tiempo)
Entender cuándo los clientes se comunican es crucial para el dimensionamiento del equipo de soporte (Staffing).

In [ ]:
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.day_name()

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Volumen por hora
sns.countplot(data=df.drop_duplicates(subset="conv_id"), x="hour", ax=axes[0], color="steelblue")
axes[0].set_title("Volumen de Conversaciones por Hora")
axes[0].set_xlabel("Hora del Día")
axes[0].set_ylabel("Cantidad")

# Volumen por día de la semana
days_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
sns.countplot(data=df.drop_duplicates(subset="conv_id"), x="day_of_week", order=days_order, ax=axes[1], color="mediumseagreen")
axes[1].set_title("Volumen de Conversaciones por Día de la Semana")
axes[1].set_xlabel("Día")
axes[1].set_ylabel("Cantidad")

plt.tight_layout()
plt.show()

## 5. Análisis de Sentimiento y Urgencia
¿Cómo se sienten los clientes y qué tan urgente es su solicitud? Analizamos estas dimensiones clave.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sentimiento
sentiment_counts = df["overall_sentiment"].value_counts()
axes[0].pie(sentiment_counts, labels=sentiment_counts.index, autopct="%1.1f%%", colors=["#e74c3c", "#95a5a6", "#2ecc71"], startangle=90)
axes[0].set_title("Distribución de Sentimiento General")

# Urgencia
sns.countplot(data=df, x="overall_urgency", order=["low", "medium", "high", "critical"], ax=axes[1], palette="Reds")
axes[1].set_title("Niveles de Urgencia")
axes[1].set_xlabel("Urgencia")

plt.tight_layout()
plt.show()

### Interacción entre Sentimiento e Industria
Verificamos qué industrias presentan mayor negatividad.

In [ ]:
industry_sentiment = pd.crosstab(df["industry"], df["overall_sentiment"], normalize="index") * 100

industry_sentiment.plot(kind="bar", stacked=True, figsize=(12, 6), color=["#e74c3c", "#95a5a6", "#2ecc71"])
plt.title("Proporción de Sentimiento por Industria")
plt.ylabel("Porcentaje (%)")
plt.xlabel("Industria")
plt.legend(title="Sentimiento", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Resultados de las Interacciones (Outcomes)
Evaluamos la eficiencia del equipo resolviendo los problemas planteados por los clientes.

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df.drop_duplicates(subset="conv_id"), y="outcome", order=df["outcome"].value_counts().index, palette="magma")
plt.title("Resultados de las Conversaciones (Outcomes)")
plt.xlabel("Cantidad")
plt.ylabel("Resultado")
plt.show()

## 7. Análisis de Intención Principal (Primary Intent)
Identificamos las razones principales por las cuales los clientes contactan al soporte.

In [ ]:
plt.figure(figsize=(12, 8))
top_intents = df["primary_intent"].value_counts().nlargest(15)
sns.barplot(x=top_intents.values, y=top_intents.index, palette="rocket")
plt.title("Top 15 Intenciones del Cliente")
plt.xlabel("Frecuencia")
plt.ylabel("Intención")
plt.show()